<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
OpenAI Agent Builder
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"]    = "M29-OpenAI-Agent-Builder"
os.environ["LANGSMITH_ENDPOINT"]   = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

In [ ]:
#@title 🛠️ openai-agents installieren{ display-mode: "form" }
install_packages([('openai-agents', 'agents')])

# 1 | Übersicht
---

***Multi-Agent Projekt*** hat gezeigt, wie man ein Multi-Agent-System **von Hand in LangGraph** baut.
**Dieses Modul** zeigt den anderen Weg: den **OpenAI Agent Builder** – ein visuelles No-Code-Werkzeug,
mit dem dieselben Workflows per Drag & Drop entworfen werden können.

Außerdem schauen wir uns den **OpenAI Agents SDK** an – die Python-Entsprechung
des visuellen Builders für Entwickler.

| Thema | M21 (LangGraph) | M22 (Agent Builder / SDK) |
|-------|-----------------|---------------------------|
| **Ansatz** | Code-first | No-Code + SDK |
| **Workflows** | Python StateGraph | Drag & Drop / `Agent` Klasse |
| **Tools** | `@tool` Decorator | Tool-Nodes / `@function_tool` |
| **Routing** | `supervisor_router()` | Condition-Nodes / LLM entscheidet |
| **Deployment** | Manuell | One-Click / Built-in |
| **Lernkurve** | Mittel | Niedrig (UI) / Gering (SDK) |


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Kurs-Einordnung M22</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    M19["M19\nMulti-Agent\nPatterns"]
    M20["M20\nSupervisor\nDeep Dive"]
    M21["M21\nMulti-Agent\nProjekt"]
    M22(["🎯 M22\nAgent Builder\n& SDK"])
    M23["M23\nAgentic RAG"]

    M19 --> M20 --> M21 --> M22 --> M23

    style M19  fill:#37474F,color:#fff
    style M20  fill:#37474F,color:#fff
    style M21  fill:#37474F,color:#fff
    style M22  fill:#FF9800,color:#000
    style M23  fill:#37474F,color:#fff
'''
mermaid(diagram, width=900)


# 2 | Was ist der Agent Builder?
---

Der **OpenAI Agent Builder** ist ein visuelles No-Code-Werkzeug zur Erstellung
komplexer KI-Workflows – vorgestellt auf dem OpenAI DevDay 2025 als Teil von **AgentKit**.

<p><font color='black' size="5">Kernfunktionen</font></p>

| Funktion | Beschreibung |
|----------|--------------|
| **Visuelle Workflows** | Drag & Drop – kein Code nötig |
| **Bedingte Logik** | Wenn-Dann-Verzweigungen zwischen Nodes |
| **Multi-Agent-Koordination** | Mehrere spezialisierte Agenten orchestrieren |
| **MCP-Integration** | 100+ Services über Model Context Protocol |
| **Versionierung** | Built-in Workflow-Versionierung & Rollback |
| **Code-Export** | TypeScript/Python-Export für Anpassungen |
| **Built-in Monitoring** | Latenz, Kosten, Fehlerrate je Step |

<p><font color='black' size="5">Zugang</font></p>

- **Voraussetzung:** ChatGPT Enterprise oder Edu Account
- **URL:** [platform.openai.com/agent-builder](https://platform.openai.com/agent-builder)
- ChatGPT Plus/Team hat **keinen** Zugang zum Agent Builder

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Agent Builder Interface-Bereiche</font> </br></p>

diag_iface = '''
%%{init: {'theme':'forest'}}%%
stateDiagram-v2
    [*] --> Templates : Start hier
    Templates --> Drafts : Anpassen
    Drafts --> Drafts : Iterieren
    Drafts --> Workflows : Veröffentlichen
    Workflows --> Drafts : Klonen / Bearbeiten
    Workflows --> [*] : Deployen
'''
mermaid(diag_iface, width=650)

<p><font color='black' size="5">Node-Typen</font></p>

Agent Builder arbeitet mit einem gerichteten Graphen aus **Nodes** (Aktionen)
und **Edges** (Verbindungen) – konzeptionell identisch mit LangGraph.

| Node-Typ | Symbol | Funktion | Beispiel |
|----------|--------|----------|----------|
| **LLM** | 🤖 | Modell-Aufruf mit Prompt | Klassifikation, Zusammenfassung |
| **Tool** | 🔧 | API-Call oder MCP-Server | JIRA-Ticket, E-Mail senden |
| **Condition** | 🔀 | Verzweigung auf Basis von Daten | Wenn Priorität > 3, dann… |
| **Human** | 👤 | Human-in-the-Loop Checkpoint | Genehmigung einholen |
| **Subworkflow** | 📦 | Verschachtelung anderer Workflows | Wiederverwendbare Sub-Prozesse |

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Node-Typen im Überblick</font> </br></p>

diag_nodes = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph NT [Node-Typen im Agent Builder]
        A["🤖 LLM Node"]
        B["🔧 Tool Node"]
        C["🔀 Condition Node"]
        D["👤 Human Node"]
        E["📦 Subworkflow Node"]
    end

    A -->|Text analysieren| OUT["Output / nächster Node"]
    B -->|Externe Aktion| OUT
    C -->|Routing-Logik| OUT
    D -->|Genehmigung| OUT
    E -->|Komplex-Logik| OUT

    style A fill:#2196F3,color:#fff
    style B fill:#FF9800,color:#000
    style C fill:#FDD835,color:#000
    style D fill:#4CAF50,color:#fff
    style E fill:#9C27B0,color:#fff
    style OUT fill:#37474F,color:#fff
'''
mermaid(diag_nodes, width=900)

# 3 | No-Code vs. Code-Ansatz
---

Für Multi-Agent-Workflows gibt es drei Ansätze – von visuell bis code-nah:

| | Agent Builder | OpenAI Agents SDK | LangGraph |
|---|---|---|---|
| **Coding erforderlich** | ❌ | ✅ Python | ✅ Python |
| **Schnelles Prototyping** | ✅ | ✅ | ⚠️ |
| **Volle Code-Kontrolle** | ⚠️ Export | ✅ | ✅ |
| **Multi-Provider (OpenAI + Anthropic)** | ❌ | ❌ | ✅ |
| **On-Premise Deployment** | ❌ | ✅ | ✅ |
| **Built-in Monitoring** | ✅ | ⚠️ | ⚠️ LangSmith |
| **Built-in Versionierung** | ✅ | ❌ | ❌ |
| **MCP-Integration** | ✅ Native | ✅ | ⚠️ Custom |
| **Human-in-the-Loop** | ✅ Node | ✅ | ✅ M19 |
| **Lernkurve** | Niedrig | Gering | Mittel |
| **Kosten (Entwicklung)** | Niedrig | Niedrig | Mittel |

> ⚠️ Code-Export aus dem Builder möglich, aber auf TypeScript/Python limitiert

**Reasoning-Modelle** erweitern das Bild: Unabhängig vom Framework stellt sich die Frage, welches **Modell** im Agenten steckt.

| | Standard-Modell | Reasoning-Modell |
|---|---|---|
| **Beispiele** | gpt-4o-mini, gpt-4o | o3-mini, o3, Claude 3.7 (extended thinking) |
| **Arbeitsweise** | Antwort direkt | Internes "Denken" vor Antwort (Chain of Thought) |
| **Staerke** | Schnell, kostenguenstig | Komplexe Logik, mehrstufige Planung |
| **Latenz** | Niedrig | Hoeher (Thinking-Phase) |
| **Kosten** | Gering | 3–10× hoeher |
| **Einsatz im Agenten** | Einfaches Routing, Extraktion, FAQ | Supervisor-Logik, Code-Debugging, Mathematik |

**Wann lohnt ein Reasoning-Modell?**
- Supervisor-Agent mit vielen Worker-Agents und komplexen Routing-Bedingungen (*Supervisor Pattern*/*Multi-Agent Projekt*)
- Tool-Auswahl bei 5+ Tools mit feinen Bedeutungsunterschieden (*Multi-Tool Agents*)
- Aufgaben, die mehrere Planungsschritte erfordern, bevor ein Tool aufgerufen wird

> **Framework-unabhaengig:** `init_chat_model("openai:o3-mini")` tauscht das Modell aus – der Graph-Code in LangGraph oder der Agent-Code bleibt unveraendert.

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Entscheidungsbaum – Builder vs. SDK vs. LangGraph</font> </br></p>

diag_ent = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    START(["❓ Ich baue einen Agenten"]) --> Q1{"Team kann Python\nprogrammieren?"}

    Q1 -->|"NEIN"| AB["✅ Agent Builder\nNo-Code Workflows"]
    Q1 -->|"JA"| Q2{"Nur OpenAI-Modelle\nausreichend?"}

    Q2 -->|"JA"| Q3{"Einfacher Agent\nohne State Machine?"}
    Q2 -->|"NEIN\n(Multi-Provider)"| LG["✅ LangGraph\n(M12–M22)"]

    Q3 -->|"JA"| SDK["✅ OpenAI Agents SDK\nPython, einfach"]
    Q3 -->|"NEIN\n(komplexe Zustände)"| LG

    AB --- N1["Drag & Drop\nMCP-Integration\nEnterprise Governance"]
    SDK --- N2["@function_tool\nHandoffs\nStreamlit-fähig"]
    LG  --- N3["StateGraph\nCheckpointing\nSupervisor Pattern"]

    style AB  fill:#10a37f,color:#fff
    style SDK fill:#FF9800,color:#000
    style LG  fill:#2196F3,color:#fff
    style N1  fill:#1B5E20,color:#fff
    style N2  fill:#E65100,color:#fff
    style N3  fill:#0D47A1,color:#fff
'''
mermaid(diag_ent, width=620)

# 4 | Live-Demo: Support-Ticket-Routing
---

**Szenario:** Eingehende Support-Tickets werden automatisch kategorisiert,
priorisiert und an die richtige Abteilung weitergeleitet.

**Anforderungen:**
- Kategorisierung: `technical` | `billing` | `sales`
- Prioritäts-Bewertung: 1–5 (5 = kritisch)
- Bedingte Weiterleitung je nach Kategorie & Priorität
- Human-in-the-Loop für Sales-Anfragen
- Bestätigungs-E-Mail an Kunden

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Support-Ticket-Routing Workflow</font> </br></p>

diag_ticket = '''
%%{init: {'theme':'dark'}}%%
flowchart TB
    START(["📨 Ticket eingehend"]) --> PARSE["🔧 Parse Ticket Data"]
    PARSE --> LLM["🤖 LLM: Analyze & Categorize\ngpt-4o-mini, temp=0.0"]

    LLM --> COND{"🔀 Category + Priority?"}

    COND -->|"technical\nPriority > 3"| JIRA["🔧 Create JIRA Ticket"]
    COND -->|"technical\nPriority ≤ 3"| QUEUE["🔧 Add to Support Queue"]
    COND -->|"billing"| FINANCE["🔧 Assign to Finance"]
    COND -->|"sales"| HUMAN["👤 Human Review Required"]

    JIRA --> EMAIL["🔧 Send Confirmation Email"]
    QUEUE --> EMAIL
    FINANCE --> EMAIL
    HUMAN --> APPR{"Approved?"}
    APPR -->|"Ja"| EMAIL
    APPR -->|"Nein"| REJECT["❌ Send Rejection Notice"]

    EMAIL --> FINISH(["✅ Workflow Complete"])
    REJECT --> FINISH

    style START  fill:#2E7D32,color:#fff
    style FINISH fill:#1A237E,color:#fff
    style LLM    fill:#2196F3,color:#fff
    style COND   fill:#FDD835,color:#000
    style HUMAN  fill:#4CAF50,color:#fff
    style APPR   fill:#FDD835,color:#000
    style REJECT fill:#B71C1C,color:#fff
    style JIRA   fill:#FF9800,color:#000
    style QUEUE  fill:#FF9800,color:#000
    style FINANCE fill:#FF9800,color:#000
    style EMAIL  fill:#FF9800,color:#000
'''
mermaid(diag_ticket, width=820)

<p><font color='black' size="5">Node-Konfigurationen im Builder</font></p>

Im Agent Builder wird jeder Node visuell konfiguriert.
So sehen die Einstellungen für die drei wichtigsten Nodes aus:

**🤖 LLM Node: `Analyze & Categorize`**
```yaml
Node Type: LLM
Model: gpt-4o-mini
Temperature: 0.0

System Prompt:
  Du bist ein Support-Ticket-Klassifizierer.
  Analysiere das Ticket und gib zurück:
  - category: technical | billing | sales
  - priority: 1–5  (1=niedrig, 5=kritisch)
  - summary: Ein-Satz-Zusammenfassung

Input:  {ticket_text}
Output: JSON {category, priority, summary}
```

**🔀 Condition Node: `Category Router`**
```yaml
Node Type: Condition
Branches:
  - IF output.category == 'technical' AND output.priority > 3
    THEN goto 'Create JIRA Ticket'
  - IF output.category == 'technical' AND output.priority <= 3
    THEN goto 'Add to Support Queue'
  - IF output.category == 'billing'
    THEN goto 'Assign to Finance'
  - IF output.category == 'sales'
    THEN goto 'Human Review Required'
```

**🔧 Tool Node: `Create JIRA Ticket` (via MCP)**
```yaml
Node Type: Tool (MCP)
MCP Server: jira
Function:   create_issue
Parameters:
  project:     'SUP'
  summary:     {output.summary}
  priority:    {output.priority}
  description: {ticket_text}
Output: {jira_id}
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: MCP-Integration im Workflow</font> </br></p>

diag_mcp = '''
%%{init: {'theme':'dark'}}%%
flowchart TB
    WF["Agent Builder Workflow"] --> MCP["MCP Protocol Layer"]

    MCP --> JIRA["JIRA Server"]
    MCP --> SLACK["Slack Server"]
    MCP --> EMAIL2["E-Mail Server"]
    MCP --> DB["PostgreSQL Server"]
    MCP --> CUSTOM["Custom MCP Server"]

    JIRA   --> JAPI["JIRA API"]
    SLACK  --> SAPI["Slack API"]
    EMAIL2 --> EAPI["SMTP / SendGrid"]
    DB     --> DAPI["Database"]
    CUSTOM --> CAPI["Ihre API"]

    style MCP    fill:#10a37f,color:#fff
    style WF     fill:#2196F3,color:#fff
    style JIRA   fill:#FF9800,color:#000
    style SLACK  fill:#9C27B0,color:#fff
    style EMAIL2 fill:#FF9800,color:#000
    style CUSTOM fill:#37474F,color:#fff
'''
mermaid(diag_mcp, width=780)

# 5 | Vergleich: Builder vs. LangGraph
---

Der Support-Ticket-Workflow aus Kapitel 4 lässt sich 1:1 in LangGraph nachbauen.
Der Vergleich zeigt: **gleiche Konzepte, andere Werkzeuge**.

<p><font color='black' size="5">Konzept-Äquivalenzen</font></p>

| Agent Builder | LangGraph / LangChain | Funktion |
|---------------|----------------------|----------|
| LLM Node | `llm.invoke()` in Node-Funktion | Modell aufrufen |
| Condition Node | `add_conditional_edges()` + Router | Bedingtes Routing |
| Tool Node | `@tool` Decorator | Externe Aktion |
| Human Node | `interrupt()` aus M19 | Human-in-the-Loop |
| Subworkflow | Nested `StateGraph` | Wiederverwendbare Logik |
| Workflow veröffentlichen | `graph.compile()` | Graph fertigstellen |
| MCP Server | `@tool` mit HTTP-Request | Service-Integration |
| Versioning | Git-Commits | Nachvollziehbarkeit |

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Builder-Konzepte in LangGraph</font> </br></p>

diag_equiv = '''
%%{init: {'theme':'forest'}}%%
flowchart LR
    subgraph AB ["Agent Builder (No-Code)"]
        AB1["🤖 LLM Node"]
        AB2["🔀 Condition Node"]
        AB3["🔧 Tool Node"]
        AB4["👤 Human Node"]
    end

    subgraph LG ["LangGraph (Python)"]
        LG1["llm.invoke()\nin Node-Funktion"]
        LG2["add_conditional_edges()\n+ Router-Funktion"]
        LG3["@tool Decorator\n+ ReAct Agent"]
        LG4["interrupt()\n(M18 HITL)"]
    end

    AB1 <-."gleichwertig".-> LG1
    AB2 <-."gleichwertig".-> LG2
    AB3 <-."gleichwertig".-> LG3
    AB4 <-."gleichwertig".-> LG4

    style AB1 fill:#10a37f,color:#fff
    style AB2 fill:#10a37f,color:#fff
    style AB3 fill:#10a37f,color:#fff
    style AB4 fill:#10a37f,color:#fff
    style LG1 fill:#2196F3,color:#fff
    style LG2 fill:#2196F3,color:#fff
    style LG3 fill:#2196F3,color:#fff
    style LG4 fill:#2196F3,color:#fff
'''
mermaid(diag_equiv, width=900)


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
from typing import Annotated, Literal
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image as IPImage

# ── Konfigurationskonstanten ───────────────────────────────────────────────
MAX_RETRIES = 3   # API-Retry-Versuche bei transienten Fehlern (with_retry)

llm = init_chat_model('openai:gpt-4o-mini', temperature=0.0)

# ── LangGraph-Äquivalent: Ticket-Klassifizierung ─────────────────────────
# (entspricht dem LLM Node + Condition Node im Agent Builder)

class TicketKlassifizierung(BaseModel):
    'Strukturierte Ausgabe des Klassifizierungs-Nodes.'
    category: Literal['technical', 'billing', 'sales'] = Field(
        description='Ticket-Kategorie'
    )
    priority: int = Field(
        description='Prioritaet 1-5', ge=1, le=5
    )
    summary: str = Field(
        description='Kurzzusammenfassung des Tickets'
    )

# .with_retry(): schützt vor transienten API-Fehlern bei der Klassifizierung
classifier_llm = (
    llm
    .with_structured_output(TicketKlassifizierung)
    .with_retry(stop_after_attempt=MAX_RETRIES)
)

# ── State ─────────────────────────────────────────────────────────────────
class TicketState(TypedDict):
    messages:      Annotated[list, add_messages]
    ticket_text:   str
    category:      str
    priority:      int
    summary:       str
    jira_id:       str

# ── Nodes ─────────────────────────────────────────────────────────────────
CLASSIFY_PROMPT = (
    'Du bist ein Support-Ticket-Klassifizierer. '
    'Analysiere das Ticket und kategorisiere es.'
)

def classify_node(state: TicketState) -> dict:
    'Entspricht dem LLM-Node im Agent Builder.'
    result = classifier_llm.invoke([
        SystemMessage(CLASSIFY_PROMPT),
        HumanMessage(f"Ticket: {state['ticket_text']}"),
    ])
    mprint(f'**🤖 Klassifizierung:** `{result.category}` | Prio {result.priority}')
    mprint(f'**📝 Summary:** {result.summary}')
    return {
        'category': result.category,
        'priority': result.priority,
        'summary':  result.summary,
    }

@tool
def jira_ticket_erstellen(summary: str, priority: int) -> str:
    'Erstellt ein JIRA-Ticket. Entspricht dem Tool-Node im Agent Builder.'
    ticket_id = f'SUP-{abs(hash(summary)) % 9000 + 1000}'
    return f'JIRA-Ticket erstellt: {ticket_id} (Prio {priority})'

@tool
def support_queue_hinzufuegen(summary: str) -> str:
    'Fuegt Ticket der Support-Queue hinzu.'
    return f'Ticket zur Support-Queue hinzugefuegt: {summary[:50]}'

@tool
def finance_zuweisen(summary: str) -> str:
    'Weist Ticket dem Finance-Team zu.'
    return f'Ticket an Finance weitergeleitet: {summary[:50]}'

def jira_node(state: TicketState) -> dict:
    result = jira_ticket_erstellen.invoke({'summary': state['summary'],
                                          'priority': state['priority']})
    mprint(f'**🔧 JIRA:** {result}')
    return {'jira_id': result}

def queue_node(state: TicketState) -> dict:
    result = support_queue_hinzufuegen.invoke({'summary': state['summary']})
    mprint(f'**🔧 Queue:** {result}')
    return {'jira_id': result}

def finance_node(state: TicketState) -> dict:
    result = finance_zuweisen.invoke({'summary': state['summary']})
    mprint(f'**🔧 Finance:** {result}')
    return {'jira_id': result}

def sales_node(state: TicketState) -> dict:
    mprint('**👤 Sales:** Human Review erforderlich – Ticket eskaliert.')
    return {'jira_id': 'HUMAN-REVIEW'}

# ── Router (entspricht dem Condition Node) ────────────────────────────────
def ticket_router(state: TicketState) -> str:
    'Entspricht dem Condition Node im Agent Builder.'
    cat  = state.get('category', 'technical')
    prio = state.get('priority', 1)
    if cat == 'technical' and prio > 3:
        return 'jira'
    elif cat == 'technical':
        return 'queue'
    elif cat == 'billing':
        return 'finance'
    else:
        return 'sales'

# ── Graph COMPILE ─────────────────────────────────────────────────────────
builder = StateGraph(TicketState)
builder.add_node('classify', classify_node)
builder.add_node('jira',    jira_node)
builder.add_node('queue',   queue_node)
builder.add_node('finance', finance_node)
builder.add_node('sales',   sales_node)

builder.add_edge(START, 'classify')
builder.add_conditional_edges(
    'classify', ticket_router,
    {'jira': 'jira', 'queue': 'queue',
     'finance': 'finance', 'sales': 'sales'}
)
for node in ['jira', 'queue', 'finance', 'sales']:
    builder.add_edge(node, END)

ticket_graph = builder.compile()
print(f'✅ Ticket-Routing-Graph kompiliert')
print(f'   classifier_llm: gpt-4o-mini + with_structured_output + with_retry(stop_after_attempt={MAX_RETRIES})')

# ── VIZ ───────────────────────────────────────────────────────────────────
try:
    display(IPImage(ticket_graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f'(Graph-Viz nicht verfuegbar: {e})')

In [ ]:
# ── Ticket-Tests ──────────────────────────────────────────────────────────
tickets = [
    ('Kritischer Produktionsfehler – API gibt 500er zurück, alle Nutzer betroffen!',
     'Erwarte: technical, Prio 5 → JIRA'),
    ('Kann meine Rechnung nicht herunterladen.',
     'Erwarte: billing → Finance'),
    ('Interesse an Enterprise-Plan – bitte Angebot zusenden.',
     'Erwarte: sales → Human Review'),
]

for ticket_text, erwartung in tickets:
    mprint(f'---\n**🎫 Ticket:** {ticket_text}')
    mprint(f'_Erwartung: {erwartung}_')
    result = ticket_graph.invoke({
        'messages':    [],
        'ticket_text': ticket_text,
        'category': '', 'priority': 0, 'summary': '', 'jira_id': '',
    })
    mprint(f'**Ergebnis:** `{result["category"]}` | Prio {result["priority"]} | `{result["jira_id"]}`\n')

# 6 | Mini-Hands-On: OpenAI Agents SDK
---

Der **OpenAI Agents SDK** ist die Python-Entsprechung des visuellen Agent Builders.
Er erlaubt dieselben Konzepte – Agents, Tools, Handoffs – in reinem Python-Code.

**Vorteile gegenüber dem visuellen Builder:**
- ✅ Volle Code-Kontrolle
- ✅ In Jupyter, Skripten und Anwendungen nutzbar
- ✅ Custom Python-Logik in Tools
- ✅ Kein Enterprise-Account nötig

**Schlüsselkonzepte:**

| Konzept | Beschreibung |
|---------|-------------|
| `Agent` | Basisklasse – Modell + Instructions + Tools |
| `@function_tool` | Decorator für Python-Funktionen als Tools |
| `Runner.run()` | Führt einen Agenten aus (async) |
| **Handoffs** | Agent delegiert an anderen Agent |
| `RunResult` | Enthält alle Schritte und die finale Antwort |

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: OpenAI Agents SDK Architektur</font> </br></p>

diag_sdk = '''
%%{init: {'theme':'dark'}}%%
flowchart TD
    USER["👤 User-Anfrage"] --> RUNNER["Runner.run(agent, input)"]

    RUNNER --> AGENT["🤖 Agent\ninstructions + model"]

    AGENT --> THINK{"Tool nötig?"}
    THINK -->|"JA"| TOOL["@function_tool\nPython-Funktion"]
    THINK -->|"NEIN"| FINAL["Finale Antwort"]
    TOOL --> OBS["Tool-Ergebnis\nbeobachten"]
    OBS --> THINK

    AGENT --> HANDOFF{"Handoff?"}
    HANDOFF -->|"JA"| AGENT2["🤖 Spezialist-Agent"]
    AGENT2 --> FINAL

    FINAL --> OUT["RunResult\n(Schritte + Antwort)"]

    style AGENT   fill:#10a37f,color:#fff
    style AGENT2  fill:#10a37f,color:#fff
    style RUNNER  fill:#2196F3,color:#fff
    style TOOL    fill:#FF9800,color:#000
    style HANDOFF fill:#FDD835,color:#000
    style THINK   fill:#FDD835,color:#000
    style OUT     fill:#37474F,color:#fff
'''
mermaid(diag_sdk, width=550)

In [ ]:
# ── Einfacher Agent mit Tool ─────────────────────────────────────────────
import asyncio
from agents import Agent, Runner, function_tool

@function_tool
def wetter_abfragen(stadt: str) -> str:
    'Gibt aktuelles Wetter fuer eine Stadt zurueck (Demo-Daten).'
    # Umlaute normalisieren: München → Muenchen, Köln → Koeln, etc.
    ersetzungen = {
        'ä': 'ae', 'ö': 'oe', 'ü': 'ue',
        'Ä': 'Ae', 'Ö': 'Oe', 'Ü': 'Ue',
    }
    stadt_norm = stadt
    for umlaut, ersatz in ersetzungen.items():
        stadt_norm = stadt_norm.replace(umlaut, ersatz)

    demo = {
        'Berlin':   '🌤️ 18°C, leicht bewoelkt',
        'Hamburg':  '🌧️ 14°C, Regen',
        'Muenchen': '☀️ 22°C, sonnig',
        'Koeln':    '⛅ 16°C, wechselhaft',
        'Frankfurt': '🌥️ 17°C, bewoelkt',
    }
    return demo.get(stadt_norm, demo.get(stadt, f'Keine Demo-Daten fuer {stadt}'))

@function_tool
def aktivitaet_vorschlagen(wetter: str) -> str:
    'Schlaegt Aktivitaeten basierend auf dem Wetter vor.'
    if 'sonnig' in wetter or '☀️' in wetter:
        return 'Perfekt fuer Fahrradfahren, Picknick oder Stadtspaziergang!'
    elif 'Regen' in wetter or '🌧️' in wetter:
        return 'Ideal fuer Museum, Kino oder gemütliches Café!'
    return 'Leichte Outdoor-Aktivitaeten oder Indoor-Optionen gleichermassen gut.'

wetter_agent = Agent(
    name='Wetter_Assistent',
    instructions=(
        'Du bist ein freundlicher Wetter-Assistent. '
        'Rufe IMMER zuerst wetter_abfragen auf, dann aktivitaet_vorschlagen. '
        'Antworte auf Deutsch.'
    ),
    model='gpt-4o-mini',
    tools=[wetter_abfragen, aktivitaet_vorschlagen],
)

print('✅ Wetter-Agent erstellt mit:')
print('   @function_tool wetter_abfragen  (Umlaut-normalisiert)')
print('   @function_tool aktivitaet_vorschlagen')

In [ ]:
# ── Agent ausführen ──────────────────────────────────────────────────────
async def frage_stellen(frage: str):
    mprint(f'**👤 Frage:** {frage}\n')
    result = await Runner.run(wetter_agent, frage)
    mprint(f'**🤖 Antwort:**\n{result.final_output}')

# In Jupyter kann await direkt genutzt werden
await frage_stellen('Wie ist das Wetter in München? Was kann ich heute unternehmen?')

In [ ]:
# ── Handoff: Multi-Agent mit Spezialisierung ─────────────────────────────
# Handoffs entsprechen den Subworkflow-Nodes im Agent Builder
# WICHTIG: Agent-Namen nur mit Buchstaben, Ziffern und Unterstrichen
#          (keine Bindestriche – ungültig für Function-Calling)

from agents import handoff

# Spezialist-Agent für technische Fragen
tech_agent = Agent(
    name='Technik_Spezialist',
    instructions=(
        'Du bist ein Experte fuer technische Probleme. '
        'Analysiere das Problem und erklaere die Loesung Schritt fuer Schritt. '
        'Antworte auf Deutsch.'
    ),
    model='gpt-4o-mini',
)

# Spezialist-Agent für Billing
billing_agent = Agent(
    name='Billing_Spezialist',
    instructions=(
        'Du bist ein Experte fuer Rechnungs- und Zahlungsfragen. '
        'Beantworte Fragen klar und hilfsbereit. '
        'Antworte auf Deutsch.'
    ),
    model='gpt-4o-mini',
)

# Triage-Agent: empfängt alle Anfragen und leitet weiter
triage_agent = Agent(
    name='Triage_Agent',
    instructions=(
        'Du bist ein Support-Triage-Agent. '
        'Leite Anfragen an den passenden Spezialisten weiter:\n'
        '- Technische Probleme → Technik_Spezialist\n'
        '- Rechnungs-/Zahlungsfragen → Billing_Spezialist\n'
        'Antworte auf Deutsch.'
    ),
    model='gpt-4o-mini',
    handoffs=[handoff(tech_agent), handoff(billing_agent)],
)

print('✅ Handoff-System erstellt:')
print('   Triage_Agent → [Technik_Spezialist, Billing_Spezialist]')

In [ ]:
# ── Handoff testen ───────────────────────────────────────────────────────
async def support_anfrage(anfrage: str):
    mprint(f'**🎫 Anfrage:** {anfrage}\n')
    result = await Runner.run(triage_agent, anfrage)
    # Den aktiven Agenten identifizieren
    letzter_agent = result.last_agent.name if result.last_agent else 'Triage'
    mprint(f'**🤖 Bearbeitet von:** `{letzter_agent}`')
    mprint(f'**Antwort:**\n{result.final_output}')

# Test 1: Technische Anfrage
await support_anfrage('Meine API gibt 429 Too Many Requests zurück. Was soll ich tun?')

In [ ]:
# Test 2: Billing-Anfrage
await support_anfrage('Ich habe zweimal für denselben Monat eine Rechnung bekommen.')

In [ ]:
#@title
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M29-OpenAI-Agent-Builder", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
Aufgabe 1: Ticket-Routing erweitern (LangGraph)
</font></p>

Erweitere den `ticket_graph` aus Kapitel 5 um eine **Bestätigungs-E-Mail**:

- Nach jedem Routing-Pfad (jira / queue / finance / sales) soll eine
  Bestätigungs-E-Mail versendet werden
- Implementiere ein `email_node` das im State `email_gesendet: bool` setzt
- Alle vier Paths (jira, queue, finance, sales) → email → END

```python
def email_node(state: TicketState) -> dict:
    mprint(f'📧 Bestätigungs-E-Mail gesendet für Ticket {state["jira_id"]}')
    return {'email_gesendet': True}
```

<p><font color='black' size="5">
Aufgabe 2: Eigener Agent mit Handoffs (Agents SDK)
</font></p>

Baue ein **Reise-Planungs-System** mit dem Agents SDK:

- **`triage_agent`** – empfängt alle Anfragen
- **`hotel_agent`** – Hotelempfehlungen (`hotel_suchen` Tool)
- **`flug_agent`** – Fluginformationen (`flug_suchen` Tool)

```python
@function_tool
def hotel_suchen(stadt: str, budget: str) -> str:
    'Gibt Hotel-Empfehlungen fuer eine Stadt und Budget zurueck.'
    ...

@function_tool
def flug_suchen(von: str, nach: str, datum: str) -> str:
    'Sucht Flugoptionen zwischen zwei Staedten.'
    ...
```

**Test-Anfrage:** *"Ich möchte vom 15.–20. Juli von Berlin nach Barcelona.
Was kostet ein Flug und wo kann ich für unter 100€/Nacht wohnen?"*

<p><font color='black' size="5">
Aufgabe 3: Builder vs. LangGraph – Eigene Entscheidung
</font></p>

Wähle **einen** der folgenden Anwendungsfälle und entscheide:
Würdest du Agent Builder, den Agents SDK oder LangGraph verwenden?

**Szenarien:**

| Szenario | Beschreibung |
|----------|--------------|
| A | Ein Marketing-Team (kein Coding) will Blogposts automatisch in Social-Media-Posts umwandeln |
| B | Eine Bank braucht On-Premise-Deployment für einen Kredit-Prüf-Agenten |
| C | Ein Startup will schnell einen MVP für einen Kunden-Support-Bot bauen |
| D | Ein Data-Science-Team will mehrere LLM-Provider vergleichen (OpenAI vs. Anthropic) |

**Aufgabe:**
1. Wähle ein Szenario
2. Begründe deine Entscheidung (Werkzeug + Warum) in einer Markdown-Zelle
3. Skizziere die Architektur mit einem Mermaid-Diagramm
4. Liste die 3 wichtigsten Vorteile deines gewählten Ansatzes auf